In [3]:
%%time
import pandas as pd
import ee
from src.gee_utils import ee_init
ee_init()


def chirps_monthly_by_regions(
    asset_regions: str,
    start: str,
    end: str,
    region_id_prop: str,
    scale_m: int = 5566,   # ~0.05° native CHIRPS
    tilescale: int = 4,
    batch_size: int = 5000,) -> pd.DataFrame:
    """
    Build a monthly CHIRPS rainfall table (mm/month) aggregated by polygons.

    Parameters
    ----------
    asset_regions : str
        GEE FeatureCollection asset path to polygons (e.g., "projects/.../rainfall_regions").
    start, end : str
        Date range; end is exclusive (YYYY-MM-DD).
    region_id_prop : str
        Feature property holding a stable region ID/name.
    scale_m : int
        Reduce scale in meters.
    tilescale : int
        tileScale for reduceRegions (controls server memory/tiling).
    batch_size : int
        Number of features to pull per client-side batch.

    Returns
    -------
    pd.DataFrame with columns: ['region_id','year','month','ym','precip_mm']
    """
    ee.Initialize()

    # Collections
    chirps_daily = (ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
                    .filterDate(start, end))
    regions = ee.FeatureCollection(asset_regions)

    # Tag each daily image with year/month + "YYYY-MM"
    def _tag_month(img):
        d = img.date()
        return img.set({
            "year":  d.get("year"),
            "month": d.get("month"),
            "ym":    d.format("YYYY-MM"),
        })
    tagged = chirps_daily.map(_tag_month)

    # Monthly sum of daily precip (mm/day -> mm/month)
    def _monthly_sum(ym):
        ym = ee.String(ym)
        ic = tagged.filter(ee.Filter.eq("ym", ym))
        summed = (ic.select("precipitation").sum()
                    .rename("precip_mm")
                    .set("ym", ym)
                    .set("year", ee.Number.parse(ym.slice(0,4)))
                    .set("month", ee.Number.parse(ym.slice(5,7))))
        return summed

    unique_yms = tagged.aggregate_array("ym").distinct().sort()
    monthly_ic = ee.ImageCollection(unique_yms.map(_monthly_sum))

    # Zonal stats (mean over polygons). Change Reducer if desired.
    def _reduce_month(img):
        feats = (img.reduceRegions(
                    collection=regions,
                    reducer=ee.Reducer.mean(),
                    scale=scale_m,
                    tileScale=tilescale
                )
                .map(lambda f: f.set({
                    "ym": img.get("ym"),
                    "year": img.get("year"),
                    "month": img.get("month"),
                })))
        return feats

    per_region_fc = monthly_ic.map(_reduce_month).flatten()

    # Pull to client in chunks -> list of dicts
    def _fc_to_rows(fc, n_batch=batch_size):
        n = fc.size().getInfo()
        rows = []
        for start_idx in range(0, n, n_batch):
            sub = (ee.FeatureCollection(fc.toList(n_batch, start_idx))
                   .aggregate_array(".all").getInfo())
            for feat in sub:
                rows.append(feat["properties"])
        return rows

    rows = _fc_to_rows(per_region_fc, n_batch=batch_size)
    if not rows:
        return pd.DataFrame(columns=["region_id","year","month","ym","precip_mm"])

    df = pd.DataFrame(rows)

    # The reducer output column is "<band>_mean" i.e., "precip_mm_mean"
    value_col = "mean" if "mean" in df.columns else None
    if value_col is None:
        # Fallback: find any *_mean column
        mean_cols = [c for c in df.columns if c.endswith("mean")]
        if not mean_cols:
            raise RuntimeError("No reducer result column found (expected 'mean').")
        value_col = mean_cols[0]

    df = (df.rename(columns={value_col: "precip_mm", region_id_prop: "region_id"})
            [["region_id", "year", "month", "ym", "precip_mm"]]
            .sort_values(["region_id", "year", "month"])
            .reset_index(drop=True))

    # Cast types
    df["year"] = df["year"].astype(int)
    df["month"] = df["month"].astype(int)
    df["precip_mm"] = pd.to_numeric(df["precip_mm"], errors="coerce")
    df['date'] = pd.to_datetime(df[ ["year", "month"]].assign(day=1))
    return df

chirps_df = chirps_monthly_by_regions(
    asset_regions="projects/ee-okavango/assets/shapes/rainfall_regions",
    start="1990-01-01",
    end="1991-01-01",  # exclusive
    region_id_prop="name",
)


CPU times: user 1.97 s, sys: 1.63 s, total: 3.6 s
Wall time: 11.4 s


In [ ]:
%%time
import ee
from src.gee_utils import ee_init
ee_init()

def export_chirps_monthly_by_regions(
    asset_regions: str,
    start: str,
    end: str,
    region_id_prop: str,
    scale_m: int = 5566,      # ~0.05° native CHIRPS
    tilescale: int = 4,
    # ---- export options ----
    to_cloud_storage: bool = False,
    drive_description: str = "chirps_monthly_by_regions",
    drive_filename_prefix: str = "chirps_monthly_by_regions",
    gcs_bucket: str = None,
    gcs_prefix: str = "chirps/monthly_by_regions"
):
    """
    Export monthly CHIRPS rainfall (mm/month) aggregated by polygons as one CSV.
    Columns: ['region_id','ym','year','month','precip_mm']
    """
    chirps_daily = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").filterDate(start, end)
    regions = ee.FeatureCollection(asset_regions)

    # Tag each daily with YYYY-MM (server-side)
    def _tag_month(img):
        d = img.date()
        return img.set({"ym": d.format("YYYY-MM")})
    tagged = chirps_daily.map(_tag_month)

    # Distinct months (server-side list of strings)
    months = tagged.aggregate_array("ym").distinct().sort()

    # Build per-month FeatureCollection (reduceRegions) and tidy props
    def _month_fc(ym):
        ym = ee.String(ym)
        ic = tagged.filter(ee.Filter.eq("ym", ym))
        img = (ic.select("precipitation").sum()
                 .rename("precip_mm")
                 .set({
                     "ym": ym,
                     "year": ee.Number.parse(ym.slice(0, 4)),
                     "month": ee.Number.parse(ym.slice(5, 7)),
                 }))

        fc = img.reduceRegions(
            collection=regions,
            reducer=ee.Reducer.mean(),
            scale=scale_m,
            tileScale=tilescale
        )

        # Create a NEW geometry-less feature with only desired props
        fc = fc.map(lambda f: ee.Feature(
            None,
            {
                "region_id": f.get(region_id_prop),
                "ym": img.get("ym"),
                "year": img.get("year"),
                "month": img.get("month"),
                "precip_mm": f.get("mean"),
            }
        ))
        return fc

    # months.map(...) -> ee.List of FeatureCollections
    fc_list = months.map(_month_fc)

    # Merge all month FCs into a single FC
    merged = ee.FeatureCollection(
        ee.List(fc_list).iterate(
            lambda fc, acc: ee.FeatureCollection(acc).merge(ee.FeatureCollection(fc)),
            ee.FeatureCollection([])
        )
    )

    selectors = ["region_id","ym","year","month","precip_mm"]

    # Configure and start export
    if to_cloud_storage:
        if not gcs_bucket:
            raise ValueError("Provide gcs_bucket when to_cloud_storage=True.")
        task = ee.batch.Export.table.toCloudStorage(
            collection=merged,
            description=drive_description,
            bucket=gcs_bucket,
            fileNamePrefix=gcs_prefix,
            fileFormat="CSV",
            selectors=selectors
        )
    else:
        task = ee.batch.Export.table.toDrive(
            collection=merged,
            description=drive_description,
            fileNamePrefix=drive_filename_prefix,
            fileFormat="CSV",
            selectors=selectors
        )
    task.start()
    return task

# Example run (Drive):
task = export_chirps_monthly_by_regions(
    asset_regions="projects/ee-okavango/assets/shapes/rainfall_regions",
    start="1990-01-01",
    end="1991-01-01",   # exclusive
    region_id_prop="name",
    scale_m=5566,
    tilescale=4,
    # to_cloud_storage=True, gcs_bucket="okavango-tif", gcs_prefix="chirps/monthly_by_region"
)
print("Started export task:", task.id)


Last edited: 2026-03-08
